# Zsyntetyzowany solver BFS/B1F

Ten notebook łączy wersje `bfs_solver_optimized(1).ipynb` oraz
`bfs_solver_profiles(4).ipynb`.

Najważniejsze zachowane i scalone usprawnienia:

- poprawna mechanika: wartość `4` jest granicą segmentu jak `-10`;
- trzy niezmienione profile kolejności: `navigator`, `combinator`, `pusher`;
- profil `pusher` rozpoznaje miecz `8` i używa relaksowanej odległości miecza do pozostałych potworów;
- kompresja plansz LZ4 z bezpiecznym fallbackiem zlib;
- deterministyczny podwójny hash Zobrista;
- lokalne odcinanie dwóch klawiszy prowadzących do identycznego następcy;
- globalne rozwijanie jednego reprezentanta tego samego stanu na tej samej głębokości;
- dowód minimalności bez zmiany podstawowego sortowania `(-pscore, counter, state)`;
- limity liczby rozwinięć i czasu;
- zapis tylko rozwiązań, których optymalność została udowodniona, bez dopisywania starych wyników.

Domyślna semantyka wyniku jest **stanowa**: jeżeli kilka ciągów klawiszy
prowadzi do identycznego stanu, zachowywany jest pierwszy reprezentant. Dzięki
temu aliasy zawijania nie mnożą przestrzeni i pliku wynikowego.

In [45]:
from __future__ import annotations

from array import array
from collections import defaultdict, deque
from functools import lru_cache
from heapq import heapify, heappop, heappush
from io import BytesIO
from pathlib import Path
from time import time
import base64
import zlib

import numpy as np

try:
    import lz4.frame as _lz4_frame

    def _pack_board(board: np.ndarray) -> bytes:
        return _lz4_frame.compress(
            board.tobytes(order="C"),
            compression_level=0,
            block_linked=True,
            store_size=True,
        )

    def _unpack_board(payload: bytes, shape: tuple[int, int]) -> np.ndarray:
        raw = _lz4_frame.decompress(payload)
        return np.frombuffer(raw, dtype=np.int8).reshape(shape)

    BOARD_CODEC = "lz4"
except ImportError:
    def _pack_board(board: np.ndarray) -> bytes:
        return zlib.compress(board.tobytes(order="C"), level=1)

    def _unpack_board(payload: bytes, shape: tuple[int, int]) -> np.ndarray:
        raw = zlib.decompress(payload)
        return np.frombuffer(raw, dtype=np.int8).reshape(shape)

    BOARD_CODEC = "zlib"

PSCORE_MODES = ("navigator", "combinator", "pusher")
PUSH_OBJECT_VALUES = frozenset((2, 8))
INTERACTIVE_VALUES = frozenset((2, 3, 8, 10))
MONSTER_VALUE = 10
SWORD_VALUE = 8
BUTTON_LOAD_VALUE = 2
MAJOR_GOAL_MONSTER_REWARD = 1_000_000_000_000.0
MAJOR_GOAL_BUTTON_2_REWARD = 1_000_000_000_000.0
_MOVE_TO_CODE = {"L": 0, "U": 1, "D": 2, "R": 3}
_CODE_TO_MOVE = ("L", "U", "D", "R")
_MASK64 = (1 << 64) - 1


def load_local_map(path):
    path = Path(path)
    suffix = path.suffix.lower()
    if suffix == ".npy":
        grid = np.load(path)
    elif suffix == ".csv":
        grid = np.loadtxt(path, delimiter=",", dtype=np.int8)
    else:
        raise ValueError("Expected .npy or .csv file.")
    if grid.ndim != 2:
        raise ValueError("Grid must be a 2D matrix.")
    return grid.astype(np.int8, copy=False)



# Reproducible test fixtures. External files are preferred during normal use;
# these compressed .npy payloads are used only when requested or when a map
# file is absent. The 1_box fixture was reconstructed from the plot embedded
# in bfs_solver_optimized(1).ipynb and verified against its recorded BFS result.
_EMBEDDED_MAPS = {'0_tutorial': 'eNrtm01Lw0AQhsc55OAf6DW3KPRiQQXx7E3x4sGTFI0oiJVUvIi/wv/qVZvWZT+ymw+Tbkf67ktDGpLdh5nZSTZkPi+uzi+vd+iN3rO7fH5bZCdpdvo4ycZpdj8rXovp882suMvL42fTp3m+OD5/mL7ki/97B5PDcXp8tD9OP9K/td2v7xiiZYszVjeu1c+klMA6isrV3DNRQspWsblUSzxjJJQsjyuuhNwW6pt7sOurlF1MIkWrzlr91LaZy4zNrvNHnZN4RnKv177sw1Xt1++/EKmPS1u3mcvuxzw/fJ2KmqaI6xPV7JklVd9Q8Kwql2+/D5c7qjl62Dsx7BWOL70/ipSh283HtvYalqtN3o7N1f6+Cy5wgUsKV9lGQp/SIWgbxcLmItFq1SSPS6+x5GVVGW8UZHMxqfU4ExGLYtNc5ZaF+dG0lxQ6ouHWhdvFJS9/sdBnQsZzKgRBEARBEARBEARBEARBEARBEARBEARB/0Byq+bA1Y1L5vfe9pdQMqynKr/sbR1XTG5/zU+ddTfhx5BNzKq6OBFv1pNV/RjXc77RfFxDe66psq/qxzj2smv9XEp/9Nhc6/GfaQNzJH/Em0zVutGh7ebjCvXuxn8bG/ePL39Wr9bG+nLHkFxur6aH3Jrd0J1y3fYKZ06btkodw4/2yG4mcGPLR7Iee+kjTPz7/bnaC8fX8HmsLu7Zks1l55MY9yObS9HZsVZHEO+OzWQ/c23+W/lyXNOLkqpX2Ki4YCFVK+a81BEng8ucrSzEh252Y1HVIWRlV1nrR4lcclfcst9QdNcP4I0Scg==', '1_box': 'eNqb7BfqGxDJyFDGUK2eklqcXKRupaBek2morqOgnpZfVFKUmBefX5SSChJ3S8wpTgWKF2ckFqQC+RpGpjoKJgaaOgq1CmQCrm+DCTAAATEqQCQbQXUMDFxAhN88Nrh5OKXZIGaBLASx8akj6EGoWcQEBLEBRmSwUlMdGxsbERHIAAs7HBFIhDoGdHVgy1G8jt08MEVAHUpKgdsEcQ1WJSiBxMABcx+OUGPAJ4kZ6AxIKQ2mBUU3ij+wmYM/zgCcNPTc', '1_maze1': 'eNrtmjtPhEAQx8fFXAifgg5NqCx8xdpOY2NhZYjHRRPjGTA2xk/hd7XV4+C8fcCCYrL7J8M0LHfF/OaxMzvwcXl9cXWzQ6/0lszz8q5ITuPk7OEgSeNksSxeiuzpdlnM8+r5efZY5qvn5X32nK/We4fHaXxytJ/G7/Efr+jzi4WFZbwIYhu4E5IubIIQnqDLBwj+sWtX/UokvKLQrbq5i8jO5xNBu2a7lijyiUDWR82BoNEw0DxUr3yIIjVj9VVAlebBWlQ2ohn5EO3b6OnyQW1/3QfuokfVTLW3rtf2vi0XXFU8GwF1WjmyEAhykbGkXX0VgZz6gMjU9zcEdd0yKWr7u4mi4QTyP0LyPQ+G7ba+9Q5jdXNB9T8EvnUUGLZngv7eBPFM2la/0QhMmyMSVF22XO2RfYBIoJ/X2iPLbx/Mmu5OrZOYe2t3dmPtq5i1rj2C0PJB7xXxMppa8nmzNwnvWUy9zdMgfh/IBCzjJyO4BGJNEAJHUdRMvbEl+Jneo3NMab4xjbkAE/BsZtwpaNj03/fvHOS1gCDBJ7C/F0PrnuRMNs9GKAT6nHI6BASW1frsEvH9FhHmWyLMuVLfhAmZgScKLCwsPss35KvGrw=='}

def _load_embedded_map(name):
    payload = _EMBEDDED_MAPS.get(name)
    if payload is None:
        raise FileNotFoundError(f"No embedded test map for {name!r}.")
    raw = zlib.decompress(base64.b64decode(payload))
    return np.load(BytesIO(raw)).astype(np.int8, copy=False)

def load_named_map(name, *, prefer_embedded=False, map_path=None):
    if map_path is not None:
        return load_local_map(map_path)
    if prefer_embedded and name in _EMBEDDED_MAPS:
        return _load_embedded_map(name)
    candidates = [
        Path("mapy") / f"{name}.npy",
        Path(f"{name}.npy"),
    ]
    if name == "1_maze1":
        candidates.extend([Path("1_maze1(2).npy"), Path("1_maze1(1).npy")])
    if name == "1_box":
        candidates.append(Path("1_box_recovered.npy"))
    for candidate in candidates:
        if candidate.exists():
            return load_local_map(candidate)
    if name in _EMBEDDED_MAPS:
        return _load_embedded_map(name)
    raise FileNotFoundError(
        f"Map {name!r} not found. Checked: " + ", ".join(map(str, candidates))
    )


In [46]:
def clear_region(grid, start):
    q = deque([start])
    visited = {start}
    changes = []
    while q:
        r, c = q.popleft()
        old_value = int(grid[r, c])
        if old_value == 3:
            grid[r, c] = 0
            changes.append((r, c, old_value, 0))
        elif old_value == 4:
            grid[r, c] = -10
            changes.append((r, c, old_value, -10))
        for dr, dc in ((-1,0),(1,0),(0,-1),(0,1)):
            nr, nc = r + dr, c + dc
            if not (0 <= nr < grid.shape[0] and 0 <= nc < grid.shape[1]):
                continue
            if (nr, nc) in visited or int(grid[nr, nc]) not in (3,4):
                continue
            visited.add((nr, nc))
            q.append((nr, nc))
    return changes


def shift_objects(segment: np.ndarray, forward: bool):
    ROCK, EMPTY, MONSTER, WEAPON, PLAYER, MUD = 6, 0, 10, 8, 42, 3
    events = []
    n = len(segment)
    result = np.where(np.isin(segment, [ROCK, MONSTER, MUD]), segment, EMPTY)
    moves = []
    player_destination = None
    for i, value in enumerate(segment):
        value = int(value)
        if value in (ROCK, EMPTY, MONSTER, MUD):
            continue
        j = (i + 1) % n if forward else (i - 1) % n
        if int(segment[j]) in (MUD, ROCK):
            return segment, [], False, None
        if int(segment[j]) == MONSTER:
            if value != WEAPON:
                return segment, [], False, None
            events.append(("kill", j))
        moves.append((j, value))
        if value == PLAYER:
            player_destination = j
    for j, value in moves:
        result[j] = value
    return result, events, True, player_destination


def shift_segment(grid, tryb, pos, hero, drow, dcol):
    if tryb == 0:
        line = grid[:, pos]
        old_hero = [hero, pos]
    else:
        line = grid[pos, :]
        old_hero = [pos, hero]

    # 4 is a fixed segment boundary exactly like -10.
    l = hero
    while l >= 0 and int(line[l]) not in (-10, 4):
        l -= 1
    l += 1
    r = hero
    while r < len(line) and int(line[r]) not in (-10, 4):
        r += 1
    r -= 1

    forward = (drow + dcol) > 0
    original_segment = line[l:r+1]
    shifted_segment, events, legal, player_index = shift_objects(original_segment, forward)
    if not legal:
        return grid, old_hero, []
    if player_index is None:
        raise RuntimeError("Player was not found in the shifted segment.")
    new_hero = [l + player_index, pos] if tryb == 0 else [pos, l + player_index]
    diff_indices = np.flatnonzero(original_segment != shifted_segment)
    if diff_indices.size == 0 and not events:
        return grid, new_hero, []

    new_grid = grid.copy()
    changes = []
    for idx in diff_indices:
        idx = int(idx)
        old_value = int(original_segment[idx])
        new_value = int(shifted_segment[idx])
        rr, cc = ((l + idx, pos) if tryb == 0 else (pos, l + idx))
        changes.append((rr, cc, old_value, new_value))
    if tryb == 0:
        new_grid[l:r+1, pos] = shifted_segment
    else:
        new_grid[pos, l:r+1] = shifted_segment
    while events:
        job, j = events.pop()
        if job == "kill":
            combo_start = (l + j, pos) if tryb == 0 else (pos, l + j)
            changes.extend(clear_region(new_grid, combo_start))
    return new_grid, new_hero, changes


def apply_customs(board, parent_board, customs):
    if not customs:
        return board, []
    new_board = board
    changes = []

    def set_cell(row, col, value):
        nonlocal new_board
        old_value = int(new_board[row, col])
        value = int(value)
        if old_value == value:
            return
        if new_board is parent_board:
            new_board = parent_board.copy()
        new_board[row, col] = value
        changes.append((row, col, old_value, value))

    for kind, entries in customs:
        if kind == "buttons":
            for button, controlled in entries:
                bx, by = button
                cx, cy = controlled
                if int(new_board[bx, by]) != 0:
                    set_cell(cx, cy, max(0, int(new_board[cx, cy])))
                else:
                    set_cell(cx, cy, -10)
        elif kind == "roll":
            for trigger, controlled in entries:
                tx, ty = trigger
                cx, cy = controlled
                if int(new_board[tx, ty]) == 42 and int(new_board[cx, cy]) == -10:
                    set_cell(cx, cy, 0)
        elif kind == "blockade":
            # Blockade istnieje od stanu początkowego.
            continue
    return new_board, changes




In [47]:
def build_navigation_map(local_map, target, edit=None):
    board = np.asarray(local_map, dtype=np.int8).copy()
    board[(board != -10) & (board != 4) & (board != 6)] = 0
    board[board == 4] = -10
    for x, y in (edit or ()):
        board[x, y] = 0
    q = deque()
    dist = {}
    for t in target:
        dist[tuple(t)] = 0
        q.append(tuple(t))
    while q:
        x, y = q.popleft()
        d = dist[(x, y)]
        for drow, dcol, _ in ((-1,0,"L"),(0,1,"U"),(0,-1,"D"),(1,0,"R")):
            z = 0 if dcol == 0 else 1
            test_board = board.copy()
            test_board[x, y] = 42
            _, ng, changes = shift_segment(
                test_board, z, y if z == 0 else x, x if z == 0 else y, drow, dcol
            )
            if not changes:
                continue
            pos = (int(ng[0]), int(ng[1]))
            if pos in dist:
                continue
            dist[pos] = d + 1
            q.append(pos)
    return dist


def _open_segment(line, position):
    left = int(position)
    while left > 0 and int(line[left - 1]) not in (-10, 4):
        left -= 1
    right = int(position)
    while right + 1 < len(line) and int(line[right + 1]) not in (-10, 4):
        right += 1
    return line[left:right + 1], left


def _count_values(values, accepted):
    return sum(int(int(value) in accepted) for value in values)


def _fallback_target_distance(x, y):
    targets = globals().get("TARGET", ())
    return float(min((abs(x-tx)+abs(y-ty) for tx,ty in targets), default=0))


def _button_positions_from_customs(customs):
    return tuple(sorted({tuple(button) for kind, entries in (customs or ()) if kind == "buttons" for button, _ in entries}))


def _count_twos_on_buttons(board, button_positions=None):
    positions = globals().get("BUTTON_POSITIONS", ()) if button_positions is None else button_positions
    return sum(int(int(board[r,c]) == BUTTON_LOAD_VALUE) for r,c in positions)


def _update_major_goal_counters(monsters_remaining, twos_on_buttons, changes, button_positions=None):
    positions = globals().get("BUTTON_POSITION_SET", frozenset()) if button_positions is None else frozenset(button_positions)
    m = int(monsters_remaining)
    b = int(twos_on_buttons)
    for row,col,old_value,new_value in changes:
        old_value, new_value = int(old_value), int(new_value)
        m += int(new_value == MONSTER_VALUE) - int(old_value == MONSTER_VALUE)
        if (row,col) in positions:
            b += int(new_value == BUTTON_LOAD_VALUE) - int(old_value == BUTTON_LOAD_VALUE)
    return m,b


def _major_goal_features(monsters_remaining, twos_on_buttons):
    initial = int(globals().get("INITIAL_MONSTERS", monsters_remaining))
    defeated = max(0, initial-int(monsters_remaining))
    on_buttons = max(0, int(twos_on_buttons))
    bonus = MAJOR_GOAL_MONSTER_REWARD*defeated + MAJOR_GOAL_BUTTON_2_REWARD*on_buttons
    return defeated,on_buttons,float(bonus)


def _mechanism_features(board, x, y, changes):
    customs = globals().get("CURRENT_CUSTOMS", ()) or ()
    button_positions=set(); controlled_positions=set(); roll_triggers=set(); roll_controls=set()
    for kind, entries in customs:
        if kind == "buttons":
            for button, controlled in entries:
                button_positions.add(tuple(button)); controlled_positions.add(tuple(controlled))
        elif kind == "roll":
            for trigger, controlled in entries:
                roll_triggers.add(tuple(trigger)); roll_controls.add(tuple(controlled))
                
    active_buttons=sum(int(int(board[r,c]) != 0) for r,c in button_positions)
    open_button_controls=sum(int(int(board[r,c]) != -10) for r,c in controlled_positions)
    open_roll_controls=sum(int(int(board[r,c]) != -10) for r,c in roll_controls)
    inactive_waypoints=[pos for pos in button_positions if int(board[pos[0],pos[1]]) == 0]
    inactive_waypoints.extend(trigger for trigger,controlled in zip(sorted(roll_triggers),sorted(roll_controls)) if int(board[controlled[0],controlled[1]]) == -10)
    mechanism_distance=min((abs(x-r)+abs(y-c) for r,c in inactive_waypoints), default=0)
    departures=[(r,c) for r,c,old,new in changes if int(old) in PUSH_OBJECT_VALUES and int(new) != int(old)]
    arrivals=[(r,c) for r,c,old,new in changes if int(new) in PUSH_OBJECT_VALUES and int(new) != int(old)]
    push_target_delta=0.0
    if button_positions and departures and arrivals:
        before=min(abs(r-br)+abs(c-bc) for r,c in departures for br,bc in button_positions)
        after=min(abs(r-br)+abs(c-bc) for r,c in arrivals for br,bc in button_positions)
        push_target_delta=float(before-after)
    inactive_buttons=[pos for pos in button_positions if int(board[pos[0],pos[1]]) == 0]
    object_positions=[]
    if inactive_buttons:
        object_positions=[(int(r),int(c)) for r,c in np.argwhere(np.isin(board, tuple(PUSH_OBJECT_VALUES)))]
    object_button_distance=0.0; useful_object_route=0.0
    if inactive_buttons and object_positions:
        object_button_distance=float(sum(min(abs(orow-brow)+abs(ocol-bcol) for orow,ocol in object_positions) for brow,bcol in inactive_buttons))
        useful_object_route=float(min(abs(x-orow)+abs(y-ocol)+abs(orow-brow)+abs(ocol-bcol) for orow,ocol in object_positions for brow,bcol in inactive_buttons))
    return {
        "button_count":len(button_positions),"control_count":len(controlled_positions)+len(roll_controls),
        "active_buttons":float(active_buttons),"open_controls":float(open_button_controls+open_roll_controls),
        "mechanism_distance":float(mechanism_distance),"push_target_delta":push_target_delta,
        "object_button_distance":object_button_distance,"useful_object_route":useful_object_route,
    }


def _sword_features(board, x, y, changes):
    sword_positions=np.argwhere(board == SWORD_VALUE)
    monster_positions=np.argwhere(board == MONSTER_VALUE)
    player_sword_distance=0.0
    sword_monster_distance=0.0
    sword_shift_distance=0.0
    if len(sword_positions):
        player_sword_distance=float(min(abs(x-int(r))+abs(y-int(c)) for r,c in sword_positions))
    if len(sword_positions) and len(monster_positions):
        sword_monster_distance=float(min(abs(int(sr)-int(mr))+abs(int(sc)-int(mc)) for sr,sc in sword_positions for mr,mc in monster_positions))
        relaxed_maps=globals().get("SWORD_NAVIGATION_MAPS", {})
        relaxed=[]
        for sr,sc in sword_positions:
            sword_pos=(int(sr),int(sc))
            for mr,mc in monster_positions:
                target_pos=(int(mr),int(mc))
                dist_map=relaxed_maps.get(target_pos)
                if dist_map is not None and sword_pos in dist_map:
                    relaxed.append(float(dist_map[sword_pos]))
        sword_shift_distance=min(relaxed) if relaxed else sword_monster_distance
    sword_departures=sum(int(int(old)==SWORD_VALUE and int(new)!=SWORD_VALUE) for _,_,old,new in changes)
    sword_arrivals=sum(int(int(new)==SWORD_VALUE and int(old)!=SWORD_VALUE) for _,_,old,new in changes)
    moved_sword=min(sword_departures,sword_arrivals)
    return player_sword_distance,sword_monster_distance,sword_shift_distance,float(np.tanh(moved_sword))


def pscore(x,y,depth,board,jump_bonus,live_zero,mode="navigator",changes=(),custom_change_count=0,monsters_remaining=None,twos_on_buttons=None,return_parts=False):
    if mode not in PSCORE_MODES:
        raise ValueError(f"Unknown mode {mode!r}; expected one of {PSCORE_MODES}")
    rows,cols=board.shape
    distance_scale=max(1.0,float(rows+cols))
    fallback_distance=_fallback_target_distance(x,y)
    nav_distance=float(navigation_map.get((x,y),fallback_distance))
    jump_signal=float(np.tanh(float(jump_bonus)/3.0))
    changes=tuple(changes or ())
    zero_progress=float(live_zero-START_ZERO)
    opened_cells=sum(int(int(old)==-10 and int(new)!=-10) for _,_,old,new in changes)
    closed_cells=sum(int(int(old)!=-10 and int(new)==-10) for _,_,old,new in changes)
    object_change_cells=sum(int(int(old) in PUSH_OBJECT_VALUES or int(new) in PUSH_OBJECT_VALUES) for _,_,old,new in changes if int(old)!=int(new))
    departures=sum(int(int(old) in PUSH_OBJECT_VALUES and int(new)!=int(old)) for _,_,old,new in changes)
    arrivals=sum(int(int(new) in PUSH_OBJECT_VALUES and int(new)!=int(old)) for _,_,old,new in changes)
    moved_objects=min(departures,arrivals)
    mechanism=_mechanism_features(board,x,y,changes)
    if monsters_remaining is None: monsters_remaining=int(np.count_nonzero(board==MONSTER_VALUE))
    if twos_on_buttons is None: twos_on_buttons=_count_twos_on_buttons(board)
    defeated_monsters,twos_on_buttons,major_goal_bonus=_major_goal_features(monsters_remaining,twos_on_buttons)
    adjacent_interactions=0; adjacent_objects=0
    for dr,dc in ((-1,0),(1,0),(0,-1),(0,1)):
        nr,nc=x+dr,y+dc
        if 0<=nr<rows and 0<=nc<cols:
            value=int(board[nr,nc]); adjacent_interactions+=int(value in INTERACTIVE_VALUES); adjacent_objects+=int(value in PUSH_OBJECT_VALUES)
    vertical,vertical_start=_open_segment(board[:,y],x)
    horizontal,horizontal_start=_open_segment(board[x,:],y)
    active_line_objects=_count_values(vertical,PUSH_OBJECT_VALUES)+_count_values(horizontal,PUSH_OBJECT_VALUES)
    line_distances=[]
    for index,value in enumerate(vertical):
        if int(value) in PUSH_OBJECT_VALUES: line_distances.append(abs((vertical_start+index)-x))
    for index,value in enumerate(horizontal):
        if int(value) in PUSH_OBJECT_VALUES: line_distances.append(abs((horizontal_start+index)-y))
    nearest_line_object=min(line_distances) if line_distances else distance_scale
    opening_signal=float(np.tanh(max(0.0,opened_cells-closed_cells)/2.0))
    object_signal=float(np.tanh(object_change_cells/4.0)); moved_signal=float(np.tanh(moved_objects/2.0))
    adjacent_object_signal=min(float(adjacent_objects),4.0)/4.0
    line_signal=float(np.tanh(active_line_objects/3.0)); proximity_signal=max(0.0,1.0-float(nearest_line_object)/distance_scale)
    active_button_signal=mechanism["active_buttons"]; open_control_signal=mechanism["open_controls"]
    mechanism_distance=mechanism["mechanism_distance"]; object_button_distance=mechanism["object_button_distance"]; useful_object_route=mechanism["useful_object_route"]
    push_target_signal=float(np.tanh(mechanism["push_target_delta"]/2.0))
    button_count=float(mechanism["button_count"]); control_count=float(mechanism["control_count"])
    mechanisms_incomplete=control_count>0 and open_control_signal<control_count
    buttons_incomplete=button_count>0 and active_button_signal<button_count

    player_sword_distance=sword_monster_distance=sword_shift_distance=moved_sword_signal=0.0
    if mode == "pusher":
        player_sword_distance,sword_monster_distance,sword_shift_distance,moved_sword_signal=_sword_features(board,x,y,changes)

    if mode == "navigator":
        profile_score=-100.0*nav_distance-1.0*float(depth)
    elif mode == "combinator":
        if mechanisms_incomplete:
            profile_score=(100_000_000.0*open_control_signal+10_000_000.0*active_button_signal-100_000.0*object_button_distance-10_000.0*mechanism_distance-1_000.0*useful_object_route-1.0*float(depth)+100.0*opening_signal+20.0*object_signal+10.0*moved_signal)
        else:
            profile_score=100_000_000.0*open_control_signal-100.0*nav_distance-1.0*float(depth)+1.0*jump_signal
    else:
        if buttons_incomplete:
            profile_score=(100_000_000.0*open_control_signal+10_000_000.0*active_button_signal-1_000_000.0*object_button_distance-10_000.0*useful_object_route-1_000.0*mechanism_distance-1.0*float(depth)+500.0*max(0.0,push_target_signal)-200.0*max(0.0,-push_target_signal)+100.0*moved_signal+20.0*line_signal+10.0*adjacent_object_signal)
        elif mechanisms_incomplete:
            profile_score=100_000_000.0*open_control_signal-100_000.0*mechanism_distance-100.0*nav_distance-1.0*float(depth)
        elif monsters_remaining and np.any(board == SWORD_VALUE):
            # Sword-oriented pusher: persistent sword-to-monster progress is more
            # important than player navigation; move bonuses only break ties.
            profile_score=(-10_000_000.0*sword_shift_distance-100_000.0*sword_monster_distance-10_000.0*player_sword_distance-100.0*nav_distance-1.0*float(depth)+5_000.0*moved_sword_signal+100.0*line_signal+20.0*proximity_signal)
        else:
            profile_score=-100.0*nav_distance-1.0*float(depth)+50.0*moved_signal+10.0*line_signal+5.0*adjacent_object_signal
    score=float(major_goal_bonus+profile_score)
    parts={"mode":mode,"score":score,"profile_score":float(profile_score),"major_goal_bonus":major_goal_bonus,"defeated_monsters":float(defeated_monsters),"monsters_remaining":float(monsters_remaining),"twos_on_buttons":float(twos_on_buttons),"depth":float(depth),"goal_distance":nav_distance,"active_buttons":active_button_signal,"open_controls":open_control_signal,"object_button_distance":object_button_distance,"player_sword_distance":player_sword_distance,"sword_monster_distance":sword_monster_distance,"sword_shift_distance":sword_shift_distance,"moved_sword":moved_sword_signal}
    return (score,parts) if return_parts else score




In [60]:
def _splitmix64(value):
    value=(value+0x9E3779B97F4A7C15)&_MASK64
    value=((value^(value>>30))*0xBF58476D1CE4E5B9)&_MASK64
    value=((value^(value>>27))*0x94D049BB133111EB)&_MASK64
    return (value^(value>>31))&_MASK64


class ZobristHasher:
    def __init__(self,shape): self.rows,self.cols=shape; self._cache={}
    def _keys(self,row,col,value):
        flat_index=row*self.cols+col; cache_key=(flat_index,int(value)); keys=self._cache.get(cache_key)
        if keys is None:
            encoded_value=int(value)&0xFFFF
            token=((flat_index+1)*0xD6E8FEB86659FD93)&_MASK64
            token^=((encoded_value+0x10001)*0xA0761D6478BD642F)&_MASK64
            keys=(_splitmix64(token^0x243F6A8885A308D3),_splitmix64(token^0x13198A2E03707344)); self._cache[cache_key]=keys
        return keys
    def hash_board(self,board):
        h1=h2=0
        for row in range(self.rows):
            for col in range(self.cols):
                k1,k2=self._keys(row,col,int(board[row,col])); h1^=k1; h2^=k2
        return h1,h2
    def update(self,board_hash,changes):
        h1,h2=board_hash
        for row,col,old_value,new_value in changes:
            if int(old_value)==int(new_value): continue
            old1,old2=self._keys(row,col,old_value); new1,new2=self._keys(row,col,new_value)
            h1^=old1^new1; h2^=old2^new2
        return h1,h2


def _hash_key(board_hash): return (int(board_hash[0])<<64)|int(board_hash[1])


def _add_parent(parent_lists,parent_sets,child_key,parent_key,move_code,reset=False):
    edge=(parent_key,int(move_code))
    if reset:
        parent_lists[child_key]=[edge]; parent_sets[child_key]={edge}; return
    edge_set=parent_sets.setdefault(child_key,set())
    if edge not in edge_set:
        edge_set.add(edge); parent_lists.setdefault(child_key,[]).append(edge)


def _reconstruct_unique_paths(start_key,target_keys,parent_lists,max_solutions=None):
    @lru_cache(maxsize=None)
    def paths_to(key):
        if key == start_key:
            return ((),)
        result=[]
        for parent_key,move_code in parent_lists.get(key,()):
            for prefix in paths_to(parent_key):
                result.append(prefix+(_CODE_TO_MOVE[move_code],))
                if max_solutions is not None and len(result)>=max_solutions:
                    return tuple(result)
        return tuple(result)
    unique={}
    truncated=False
    for key in sorted(target_keys):
        for path in paths_to(key):
            text="".join(path)
            unique.setdefault(text,list(path))
            if max_solutions is not None and len(unique)>=max_solutions:
                truncated=True; break
        if truncated: break
    return [unique[text] for text in sorted(unique)],truncated


def _configure_search_globals(local_map,start,target,lim,customs):
    global TARGET,INITIAL_LIMIT,START_ZERO,CURRENT_CUSTOMS,INITIAL_MONSTERS,BUTTON_POSITIONS,BUTTON_POSITION_SET,SWORD_NAVIGATION_MAPS
    TARGET=tuple(map(tuple,target)); INITIAL_LIMIT=int(lim); START_ZERO=int(np.count_nonzero(local_map==0)); CURRENT_CUSTOMS=customs or []
    BUTTON_POSITIONS=_button_positions_from_customs(CURRENT_CUSTOMS); BUTTON_POSITION_SET=frozenset(BUTTON_POSITIONS)
    INITIAL_MONSTERS=int(np.count_nonzero(local_map==MONSTER_VALUE))
    monster_positions=[tuple(map(int,pos)) for pos in np.argwhere(local_map==MONSTER_VALUE)]
    SWORD_NAVIGATION_MAPS={pos: build_navigation_map(local_map,[pos],[]) for pos in monster_positions}


def bfs(local_map,start,target,lim=500,viz=False,customs=None,max_expanded=None,max_seconds=None,max_solutions=None):
    customs=customs or []
    local_map = apply_static_customs(
    local_map,
    customs,
    start=start,
    target=target,)
    started=time(); states=0; status="frontier_empty"
    local_map=np.asarray(local_map,dtype=np.int8); board0=local_map.copy(); board0[start[0],start[1]]=42; shape=board0.shape
    _configure_search_globals(local_map,start,target,lim,customs)
    hasher=ZobristHasher(shape); start_hash=hasher.hash_board(board0); start_key=_hash_key(start_hash)
    start_monsters=INITIAL_MONSTERS; start_twos=_count_twos_on_buttons(board0,BUTTON_POSITIONS)
    q=deque([(start[0],start[1],0,_pack_board(board0),start_hash,start_key,START_ZERO,start_monsters,start_twos)])
    best_depth={start_key:0}; expanded_depth={}; parent_lists={start_key:[]}; parent_sets={start_key:set()}; best_solution_depth=None; target_keys=set()
    while q:
        if max_seconds is not None and time()-started>=max_seconds: status="time_limit"; break
        if max_expanded is not None and states>=max_expanded: status="state_limit"; break
        x,y,depth,payload,board_hash,key,live_zero,monsters_remaining,twos_on_buttons=q.popleft()
        if best_depth.get(key)!=depth or expanded_depth.get(key)==depth: continue
        if best_solution_depth is not None and depth>=best_solution_depth: continue
        board=_unpack_board(payload,shape)
        lb=navigation_map.get((x,y))
        active_bound=best_solution_depth if best_solution_depth is not None else lim
        if lb is not None and depth+lb>active_bound: continue
        expanded_depth[key]=depth; states+=1
        generated_successors=set()
        for drow,dcol,move in ((-1,0,"L"),(0,1,"U"),(0,-1,"D"),(1,0,"R")):
            z=0 if dcol==0 else 1
            new_map,ng,changes=shift_segment(board,z,y if z==0 else x,x if z==0 else y,drow,dcol)
            new_map,custom_changes=apply_customs(new_map,board,customs); changes.extend(custom_changes)
            if not changes: continue
            nd=depth+1
            if nd>lim or (best_solution_depth is not None and nd>best_solution_depth): continue
            new_hash=hasher.update(board_hash,changes); new_key=_hash_key(new_hash)
            if new_key in generated_successors: continue
            generated_successors.add(new_key)
            old_depth=best_depth.get(new_key); move_code=_MOVE_TO_CODE[move]
            if old_depth is None or nd<old_depth:
                best_depth[new_key]=nd; _add_parent(parent_lists,parent_sets,new_key,key,move_code,reset=True); is_new_best=True
            elif nd==old_depth:
                # The same board at the same depth is a duplicate search state.
                # Keep the first representative path and do not enqueue/expand it again.
                continue
            else: continue
            is_target=tuple(ng) in TARGET
            if is_target:
                if best_solution_depth is None or nd<best_solution_depth:
                    best_solution_depth=nd; target_keys={new_key}
                elif nd==best_solution_depth: target_keys.add(new_key)
                continue
            if is_new_best and (best_solution_depth is None or nd<best_solution_depth):
                new_live_zero=live_zero+sum(int(int(new)==0)-int(int(old)==0) for _,_,old,new in changes)
                new_monsters,new_twos=_update_major_goal_counters(monsters_remaining,twos_on_buttons,changes)
                q.append((ng[0],ng[1],nd,_pack_board(new_map),new_hash,new_key,new_live_zero,new_monsters,new_twos))
    else:
        status="solved" if best_solution_depth is not None else "frontier_empty"
    optimality_proven=status in ("solved","frontier_empty") and best_solution_depth is not None
    answers,truncated=_reconstruct_unique_paths(start_key,target_keys,parent_lists,max_solutions=max_solutions) if target_keys else ([],False)
    elapsed=time()-started
    bfs.last_stats={"algorithm":"bfs","status":status,"elapsed":elapsed,"solutions":len(answers),"length":best_solution_depth,"states":states,"hash_entries":len(best_depth),"expanded_entries":len(expanded_depth),"board_codec":BOARD_CODEC,"solutions_truncated":truncated}
    print(f"\n--- BFS: {elapsed:.3f}s ---\nSTATUS: {status}\nOPTIMALITY PROVEN: {optimality_proven}\nLEN: {best_solution_depth}\n\n/STATES VISITED: {states}")
    return answers


def b1f(local_map,start,target,lim=500,viz=False,customs=None,mode="navigator",first_solution=False,max_expanded=None,max_seconds=None,max_solutions=None):
    customs=customs or []
    local_map = apply_static_customs(
    local_map,
    customs,
    start=start,
    target=target,)
    started=time(); states=0; status="frontier_empty"
    local_map=np.asarray(local_map,dtype=np.int8); board0=local_map.copy(); board0[start[0],start[1]]=42; shape=board0.shape
    _configure_search_globals(local_map,start,target,lim,customs)
    hasher=ZobristHasher(shape); start_hash=hasher.hash_board(board0); start_key=_hash_key(start_hash)
    start_monsters=INITIAL_MONSTERS; start_twos=_count_twos_on_buttons(board0,BUTTON_POSITIONS)
    start_state=(start[0],start[1],0,_pack_board(board0),start_hash,start_key,START_ZERO,start_monsters,start_twos)
    pq=[]; counter=0
    start_priority=-pscore(start[0],start[1],0,board0,0,START_ZERO,mode=mode,monsters_remaining=start_monsters,twos_on_buttons=start_twos)
    heappush(pq,(start_priority,counter,start_state))

    # One active heap entry per board key. Superseded entries stay in heap but
    # are marked stale, so they do not affect proof of optimality.
    active_token={start_key:counter}
    active_depth={start_key:0}
    frontier_depth_counts=defaultdict(int); frontier_depth_counts[0]=1

    best_depth={start_key:0}; expanded_depth={}; parent_lists={start_key:[]}; parent_sets={start_key:set()}
    best_solution_depth=None; target_keys=set(); first_solution_depth=None; first_solution_states=None
    reopened_expansions=0

    def remove_active(key,token):
        if active_token.get(key)!=token:
            return False
        depth_value=active_depth.pop(key)
        del active_token[key]
        frontier_depth_counts[depth_value]-=1
        return True

    def enqueue(priority,state):
        nonlocal counter
        key=state[5]; depth_value=state[2]
        old_token=active_token.get(key)
        if old_token is not None:
            old_depth=active_depth[key]
            frontier_depth_counts[old_depth]-=1
        counter+=1
        active_token[key]=counter; active_depth[key]=depth_value
        frontier_depth_counts[depth_value]+=1
        heappush(pq,(priority,counter,state))

    def shallower_frontier_exists():
        if best_solution_depth is None:
            return True
        return any(count>0 for depth_value,count in frontier_depth_counts.items() if depth_value<best_solution_depth)

    while pq:
        # With heuristic ordering, minimality is proved only when no active
        # frontier state shallower than the best target remains.
        if best_solution_depth is not None and not shallower_frontier_exists():
            status="solved"
            break
        if max_seconds is not None and time()-started>=max_seconds:
            status="time_limit"; break
        if max_expanded is not None and states>=max_expanded:
            status="state_limit"; break

        _,token,state=heappop(pq)
        x,y,depth,payload,board_hash,key,live_zero,monsters_remaining,twos_on_buttons=state
        if not remove_active(key,token):
            continue
        if best_depth.get(key)!=depth:
            continue
        previous_expanded=expanded_depth.get(key)
        if previous_expanded is not None and previous_expanded<=depth:
            continue
        if previous_expanded is not None and depth<previous_expanded:
            reopened_expansions+=1
        if best_solution_depth is not None and depth>=best_solution_depth:
            continue

        board=_unpack_board(payload,shape)
        lb=navigation_map.get((x,y))
        active_bound=best_solution_depth if best_solution_depth is not None else lim
        if lb is not None and depth+lb>active_bound:
            continue

        expanded_depth[key]=depth; states+=1
        generated_successors=set()
        for drow,dcol,move in ((-1,0,"L"),(0,1,"U"),(0,-1,"D"),(1,0,"R")):
            z=0 if dcol==0 else 1
            new_map,ng,changes=shift_segment(board,z,y if z==0 else x,x if z==0 else y,drow,dcol)
            new_map,custom_changes=apply_customs(new_map,board,customs); changes.extend(custom_changes)
            if not changes: continue
            #nd=depth+1
            #if nd>lim or (best_solution_depth is not None and nd>best_solution_depth): continue
            #new_hash=hasher.update(board_hash,changes); new_key=_hash_key(new_hash)
            #if new_key in generated_successors: continue
            nd = depth + 1

            active_bound = (
                best_solution_depth
                if best_solution_depth is not None
                else lim
            )
            # Sam koszt dojścia przekracza aktualne ograniczenie.
            if nd > active_bound:
                continue
            
            # Odrzucenie dziecka, zanim utworzymy hash,
            # rodziców, skompresowaną planszę i wpis w kopcu.
            lb_child = navigation_map.get(tuple(ng))
            
            if (
                lb_child is not None
                and nd + lb_child > active_bound
            ):
                continue
            
            new_hash = hasher.update(
                board_hash,
                changes,
            )
            new_key = _hash_key(new_hash)

            if new_key in generated_successors:
                continue

            generated_successors.add(new_key)

            old_depth=best_depth.get(new_key); move_code=_MOVE_TO_CODE[move]
            if old_depth is None or nd<old_depth:
                best_depth[new_key]=nd
                _add_parent(parent_lists,parent_sets,new_key,key,move_code,reset=True)
            elif nd == old_depth:
                _add_parent(
                    parent_lists,
                    parent_sets,
                    new_key,
                    key,
                    move_code,
                    reset=False,
                )
            else:
                continue

            is_target=tuple(ng) in TARGET
            if is_target:
                if first_solution_depth is None:
                    first_solution_depth=nd; first_solution_states=states
                if best_solution_depth is None or nd<best_solution_depth:
                    best_solution_depth=nd; target_keys={new_key}
                    if first_solution:
                        status="first_solution"; pq.clear(); break
                elif nd==best_solution_depth:
                    target_keys.add(new_key)
                continue

            if best_solution_depth is not None and nd>=best_solution_depth:
                continue

            travel=abs(ng[0]-x)+abs(ng[1]-y); jump_bonus=max(0,travel-1)
            new_live_zero=live_zero+sum(int(int(new)==0)-int(int(old)==0) for _,_,old,new in changes)
            new_monsters,new_twos=_update_major_goal_counters(monsters_remaining,twos_on_buttons,changes)
            child=(ng[0],ng[1],nd,_pack_board(new_map),new_hash,new_key,new_live_zero,new_monsters,new_twos)
            priority=-pscore(ng[0],ng[1],nd,new_map,jump_bonus,new_live_zero,mode=mode,changes=changes,custom_change_count=len(custom_changes),monsters_remaining=new_monsters,twos_on_buttons=new_twos)
            enqueue(priority,child)
        if status=="first_solution":
            break
    else:
        status="solved" if best_solution_depth is not None else "frontier_empty"

    optimality_proven=status=="solved" and best_solution_depth is not None
    answers,truncated=_reconstruct_unique_paths(start_key,target_keys,parent_lists,max_solutions=max_solutions) if target_keys else ([],False)
    elapsed=time()-started
    b1f.last_stats={"algorithm":"b1f","mode":mode,"status":status,"optimality_proven":optimality_proven,"elapsed":elapsed,"solutions":len(answers),"length":best_solution_depth,"states":states,"hash_entries":len(best_depth),"expanded_entries":len(expanded_depth),"reopened_expansions":reopened_expansions,"first_solution_depth":first_solution_depth,"first_solution_states":first_solution_states,"board_codec":BOARD_CODEC,"solutions_truncated":truncated,"initial_monsters":INITIAL_MONSTERS,"button_positions":len(BUTTON_POSITIONS)}
    print(f"\n--- B1F/{mode}: {elapsed:.3f}s ---\nSTATUS: {status}\nOPTIMALITY PROVEN: {optimality_proven}\nFIRST SOLUTION: {first_solution_depth} after {first_solution_states} states\nREOPENED EXPANSIONS: {reopened_expansions}\n\n/STATES VISITED: {states}")
    return answers

b1s=b1f


In [64]:
BLOCKADE_VALUE = 6

def _blockade_positions_from_customs(customs):
    positions = []
    seen = set()

    for kind, entries in (customs or ()):
        if kind != "blockade":
            continue

        for position in entries:
            if len(position) != 2:
                raise ValueError(
                    f"Invalid blockade position: {position!r}"
                )

            pos = tuple(map(int, position))

            if pos not in seen:
                seen.add(pos)
                positions.append(pos)

    return tuple(positions)


def apply_static_customs(
    local_map,
    customs,
    *,
    start=None,
    target=(),
):
    """
    Nanosi statyczne elementy konfiguracji przed rozpoczęciem
    wyszukiwania i obliczeniem początkowego hasha.
    """
    board = np.asarray(local_map, dtype=np.int8)
    blockades = _blockade_positions_from_customs(customs)

    if not blockades:
        return board

    rows, cols = board.shape
    target_set = set(map(tuple, target or ()))
    start_pos = tuple(start) if start is not None else None

    for row, col in blockades:
        if not (0 <= row < rows and 0 <= col < cols):
            raise ValueError(
                f"Blockade {(row, col)} is outside board {board.shape}"
            )

        if start_pos == (row, col):
            raise ValueError(
                f"Blockade cannot cover start position {(row, col)}"
            )

        if (row, col) in target_set:
            raise ValueError(
                f"Blockade cannot cover target position {(row, col)}"
            )

        old_value = int(board[row, col])

        if old_value not in (0, BLOCKADE_VALUE):
            raise ValueError(
                f"Blockade {(row, col)} would overwrite value "
                f"{old_value}; expected 0 or 6"
            )

    # Funkcja jest idempotentna — nie kopiuje planszy,
    # jeżeli wszystkie barykady są już naniesione.
    if all(
        int(board[row, col]) == BLOCKADE_VALUE
        for row, col in blockades
    ):
        return board

    board = board.copy()

    for row, col in blockades:
        board[row, col] = BLOCKADE_VALUE

    return board

def _prepare_run(
    i,
    *,
    prefer_embedded=False,
    map_path=None,
):
    name, start, target, limit, customs = files[i]

    local_map = load_named_map(
        name,
        prefer_embedded=prefer_embedded,
        map_path=map_path,
    )

    # Blockade musi istnieć przed zbudowaniem navigation_map.
    local_map = apply_static_customs(
        local_map,
        customs,
        start=start,
        target=target,
    )

    # Tylko dynamicznie otwierane pola są usuwane
    # w relaksowanej mapie nawigacyjnej.
    edit = []

    for kind, entries in (customs or ()):
        if kind in ("buttons", "roll"):
            for _, controlled in entries:
                edit.append(tuple(controlled))

        elif kind == "blockade":
            # Barykady pozostają jako 6 w navigation_map.
            continue

        else:
            raise ValueError(
                f"Unknown custom type: {kind!r}"
            )

    global navigation_map

    navigation_map = build_navigation_map(
        local_map,
        target,
        edit,
    )

    return local_map, start, target, limit, customs


def validate_path(
    local_map,
    start,
    target,
    path,
    customs=None,
):
    local_map = apply_static_customs(
        local_map,
        customs,
        start=start,
        target=target,
    )

    board = np.asarray(
        local_map,
        dtype=np.int8,
    ).copy()

    position = list(start)
    board[position[0], position[1]] = 42
    for step, move in enumerate(path, 1):
        drow, dcol = {
            "L": (-1, 0), "U": (0, 1),
            "D": (0, -1), "R": (1, 0),
        }[move]
        axis = 0 if dcol == 0 else 1
        new_board, new_position, changes = shift_segment(
            board,
            axis,
            position[1] if axis == 0 else position[0],
            position[0] if axis == 0 else position[1],
            drow,
            dcol,
        )
        new_board, custom_changes = apply_customs(
            new_board, board, customs or []
        )
        changes.extend(custom_changes)
        if not changes:
            return {
                "valid": False,
                "failed_step": step,
                "failed_move": move,
                "position": tuple(position),
            }
        board = new_board
        position = new_position
    return {
        "valid": tuple(position) in set(map(tuple, target)),
        "failed_step": None,
        "position": tuple(position),
        "monsters_remaining": int(np.count_nonzero(board == MONSTER_VALUE)),
    }


def _unique_minimal_texts(paths):
    texts = {"".join(path) for path in paths}
    if not texts:
        return []
    length = min(map(len, texts))
    return sorted(text for text in texts if len(text) == length)


def run(
    i=-1,
    mode="navigator",
    compare_bfs=False,
    show_map=False,
    save=True,
    max_expanded=None,
    max_seconds=None,
    prefer_embedded=False,
    map_path=None,
    first_solution=False,
):
    print(f"# {i} mode: {mode}")
    local_map, start, target, limit, customs = _prepare_run(
        i,
        prefer_embedded=prefer_embedded,
        map_path=map_path,
    )
    if show_map:
        import matplotlib.pyplot as plt
        plt.imshow(np.rot90(local_map))
        plt.title(files[i][0])
        plt.show()

    attempts = []
    if compare_bfs:
        bfs_paths = bfs(
            local_map, start, target, limit,
            customs=customs,
            max_expanded=max_expanded,
            max_seconds=max_seconds,
        )
        attempts.append(("bfs", bfs_paths, dict(bfs.last_stats)))

    profile_paths = b1f(
        local_map, start, target, limit,
        customs=customs,
        mode=mode, first_solution=first_solution,
        max_expanded=max_expanded,
        max_seconds=max_seconds,
    )
    attempts.append((mode, profile_paths, dict(b1f.last_stats)))

    proven_paths = [
        path
        for _, paths, stats in attempts
        if stats.get("optimality_proven")
        for path in paths
    ]
    candidate_paths = [path for _, paths, _ in attempts for path in paths]
    selected = _unique_minimal_texts(proven_paths or candidate_paths)
    proven = bool(proven_paths)

    output_path = None
    if save and selected and proven:
        output_dir = Path("solutions")
        output_dir.mkdir(exist_ok=True)
        output_path = output_dir / f"{i}_{files[i][0]}.txt"
        output_path.write_text("\n".join(selected) + "\n", encoding="utf-8")
        written = output_path.read_text(encoding="utf-8").splitlines()
        if written != selected or len(written) != len(set(written)):
            raise RuntimeError("Saved solution file failed uniqueness verification.")
    elif save and selected and not proven:
        print("Not saving: the state/time limit prevented proof of optimality.")

    run.last_report = {
        "run": i,
        "mode": mode,
        "attempts": attempts,
        "selected_count": len(selected),
        "selected_length": len(selected[0]) if selected else None,
        "optimality_proven": proven,
        "output_path": str(output_path) if output_path else None,
    }
    print("Selected solutions:", len(selected))
    print("Selected length:", len(selected[0]) if selected else None)
    #print("Optimality proven:", proven)
    if output_path:
        print("Output:", output_path)
    return selected


def _validation_case(i, algorithm, mode, max_expanded, max_seconds):
    local_map, start, target, limit, customs = _prepare_run(
        i, prefer_embedded=True
    )
    if algorithm == "bfs":
        paths = bfs(
            local_map, start, target, limit,
            customs=customs,
            max_expanded=max_expanded,
            max_seconds=max_seconds,
            max_solutions=1000,
        )
        stats = dict(bfs.last_stats)
    else:
        paths = b1f(
            local_map, start, target, limit,
            customs=customs,
            mode=mode,
            max_expanded=max_expanded,
            max_seconds=max_seconds,
            max_solutions=1000,
        )
        stats = dict(b1f.last_stats)
    validation = validate_path(
        local_map,
        start,
        target,
        paths[0] if paths else [],
        customs,
    ) if paths else {"valid": False}
    return {
        "run": i,
        "approach": algorithm if algorithm == "bfs" else mode,
        "status": stats.get("status"),
        "optimality_proven": stats.get("optimality_proven"),
        "best_length": stats.get("length"),
        "solutions": stats.get("solutions"),
        "states": stats.get("states"),
        "first_solution_length": stats.get("first_solution_depth"),
        "first_solution_states": stats.get("first_solution_states"),
        "reopened": stats.get("reopened_expansions", 0),
        "elapsed_s": round(stats.get("elapsed", 0.0), 3),
        "path_valid": validation.get("valid", False),
    }


def run_validation_suite(max_expanded=100_000, max_seconds=120):
    tests = [
        (0, "bfs", None),
        (0, "b1f", "navigator"),
        (0, "b1f", "combinator"),
        (0, "b1f", "pusher"),
        (16, "bfs", None),
        (16, "b1f", "pusher"),
        (19, "b1f", "combinator"),
    ]
    rows = []
    for i, algorithm, mode in tests:
        print("\n" + "=" * 72)
        print(f"VALIDATION: run({i}) / {algorithm if algorithm == 'bfs' else mode}")
        row = _validation_case(
            i, algorithm, mode, max_expanded, max_seconds
        )
        if row["states"] > max_expanded:
            raise AssertionError("Expanded-state limit was exceeded.")
        rows.append(row)
    frame = __import__("pandas").DataFrame(rows)
    return frame


In [261]:
INF = 500
files = [
    ("0_tutorial", (8,0), [(4,59)], 80, None),
    ("1_to_mag", (35,27), [(72,7),(72,11)], 40, None),
    ("1_to_mag", (72,11), [(81,4)], 15, None),
    ("1_cave_sword_shrink", (6,3), [(27,10)], 50, None),
    ("1_cave_sword2", (25,9), [(11,1),(9,1)], 40, None),
    ("1_cave_sword_final", (11,1), [(4,10)], 50, None),
    ("1_to_major", (36,12), [(16,17)], 60, None),
    ("1_major1", (7,3), [(20,15)], 50, [("buttons", [[(12,8),(14,9)]])]),
    ("1_major2", (20,15), [(6,21)], 50, [("buttons", [[(18,20),(16,21)],[(22,22),(15,21)]])]),
    ("1_major_final", (6,21), [(4,13)], 40, [("buttons", [[(5,16),(8,15)]])]),
    ("1_to_major", (13,17), [(9,4)], 60, None),
    ("1_to_nimbus", (20,4), [(0,13)], 60, None),
    ("1_to_nimbus2", (5,1), [(3,11)], 60, None),
    ("1_to_nimbus2", (3,11), [(5,0)], 60, None),
    ("1_backtown1", (0,13), [(12,13)], 60, None),
    ("1_to_mag", (39,9), [(41,19),(41,22)], 60, None),
    ("1_box", (21,27), [(5,21),(5,24)], 80, None),
    ("1_key1", (7,17), [(13,29),(13,30)], 80, None),
    ("1_maze1", (18,11), [(13,26)], 30, None),
    #19+
    ("1_maze1", (13,26), [(13,41)], 110,
     [("buttons", [[(7,31),(5,33)],[(20,36),(18,38)],[(7,31),(9,35)]])]),
    
    ("1_maze1", (13,41), [(25,48)], 47, #20
     [("buttons", [[(16,50),(22,48)],[(17,48),(22,47)],[(18,50),(20,49)]]),
      ("roll", [[(12,50),(12,49)]])]),
    #-------------------------#
    ("1_maze1", (24,48), [(32,58)], 25, #21
     [("buttons", [[(31,53),(27,54)]]),("blockade",[(25,47)])]),
    
    ("1_maze2", (7,9), [(27,22)], 72, #22
     [("buttons", [[(9,15),(11,16)],[(9,15),(12,16)],[(12,17),(15,19)],[(12,17),(15,20)],[(14,19),(16,18)],[(14,19),(17,18)]]),
      ("roll", [[(11,7),(11,6)]])
     ]),

    ("1_bell1", (18,31), [(44,10)], 50, #23
     [
      ("roll", [[(25,27),(24,27)]])
     ]),

    ("1_bell1", (44,10), [(20,9)], 65, #24
     [
      ("blockade",[(17,4),(18,4)])
     ]),

    ("1_bell2", (22,11), [(16,43)], 78, #25
     [
     ]),
    (26),
    (27),
    #UUURUURULDDDLDLULUUULDLULDDDRDDRRDDRDRDDRRDLDDRDLLLLDD
    #------------BOSS----------#
    #!!!

    #WORLD2
    ("2_wild_road1", (13,1), [(112,69)], INF, None),
    ("2_wild_road2", (2,7), [(83,15)], 155, None), #29
    ("2_wild_road2", (83,15), [(79,26)], 28, [
      ("blockade",[(83,14)])
     ]), #30
    ("3_to_shrub", (38,27), [(55,12),(55,16),(56,12),(56,16)], INF, [
      ("blockade",[(65,20)])
     ]), #31
    (32),
    ("3_shrub2", (66,22), [(51,34),(53,34)], 58,None),
    ("3_summit", (54,15), [(52,36),(53,36),(52,33),(53,33)], INF,None), #34
    #LUU LULLDDLLL    LDDRURDRURDR
    ("3_summit", (27,36), [(7,31)], 20,None), #35
    ("4_darkside", (132,29), [(99,30)], 45,None), #36
    ("4_darkside", (164,64), [(91,62)], 87,None), #37
    #LLUUUUUUUULLUR UULLLURU LUUUU LLLL DDLLLDRURDLLURDLLDLDDRDRRRDRDLDRDLRDRDRULL
    #LDRURULLLLUUUUUUURRRUUURRDDRRRRUUUURURUUULLLLUULUULLDLDDLLLDDLUULRRUUUUUURRDRRR
    #DDRRRRRDRRRUURRRRUUULLUUULLDLLLUUULLLUUULUUURUUULUURUUUUUULLLLLLLLLL
    #UUUUUUUUUUUUUUULUUURDRDLDDRRDLLLL
    ("5_start", (65,29), [(48,21),(48,16)], 85,[
      ("roll", [[(52,31),(53,31)]]),
        ("roll", [[(28,34),(28,33)]]),
        ("roll", [[(31,29),(32,29)]]),
        ("roll", [[(32,25),(33,25)]])
     ]), #38
    #UUULLUR RRRDLURUUU
    
    


]

In [263]:
run(38,mode="navigator",max_expanded=1_500_000,compare_bfs=0,first_solution=0);

# 38 mode: navigator

--- B1F/navigator: 175.964s ---
STATUS: solved
OPTIMALITY PROVEN: True
FIRST SOLUTION: 83 after 172100 states
REOPENED EXPANSIONS: 158972

/STATES VISITED: 226802
Selected solutions: 18
Selected length: 83
Output: solutions\38_5_start.txt
